In [ ]:
import pickle
import numpy as np
import tensorflow
from keras.models import Sequential
from keras.layers import Dense

In [ ]:
with open("/content/drive/MyDrive/Advanced AI/train_qa.txt","rb") as fp:
  train_data=pickle.load(fp)


In [ ]:
with open("/content/drive/MyDrive/Advanced AI/test_qa.txt","rb") as fp:
  test_data=pickle.load(fp)

In [ ]:
' '.join(train_data[0][0])

'Mary moved to the bathroom . Sandra journeyed to the bedroom .'

In [ ]:
' '.join(train_data[0][1])

'Is Sandra in the hallway ?'

In [ ]:
' '.join(train_data[0][2])

'n o'

In [ ]:
type(test_data)

list

In [ ]:
len(test_data)

1000

In [ ]:
len(train_data)

10000

In [ ]:
train_data[0]

(['Mary',
  'moved',
  'to',
  'the',
  'bathroom',
  '.',
  'Sandra',
  'journeyed',
  'to',
  'the',
  'bedroom',
  '.'],
 ['Is', 'Sandra', 'in', 'the', 'hallway', '?'],
 'no')

In [ ]:
vocab = set()
all_data = train_data + test_data

In [ ]:
for story, question,answer in all_data:
  vocab = vocab.union(set(story))
  vocab = vocab.union(set(question))

In [ ]:
vocab.add('no')
vocab.add('yes')

In [ ]:
vocab

{'.',
 '?',
 'Daniel',
 'Is',
 'John',
 'Mary',
 'Sandra',
 'apple',
 'back',
 'bathroom',
 'bedroom',
 'discarded',
 'down',
 'dropped',
 'football',
 'garden',
 'got',
 'grabbed',
 'hallway',
 'in',
 'journeyed',
 'kitchen',
 'left',
 'milk',
 'moved',
 'no',
 'office',
 'picked',
 'put',
 'the',
 'there',
 'to',
 'took',
 'travelled',
 'up',
 'went',
 'yes'}

In [ ]:
vocab_len = len(vocab) + 1
max_story_len = max([len(data[0]) for data in all_data])
max_story_len

156

In [ ]:
max_question_len = max([len(data[1]) for data in all_data])
max_question_len

6

In [ ]:
vocab

{'.',
 '?',
 'Daniel',
 'Is',
 'John',
 'Mary',
 'Sandra',
 'apple',
 'back',
 'bathroom',
 'bedroom',
 'discarded',
 'down',
 'dropped',
 'football',
 'garden',
 'got',
 'grabbed',
 'hallway',
 'in',
 'journeyed',
 'kitchen',
 'left',
 'milk',
 'moved',
 'no',
 'office',
 'picked',
 'put',
 'the',
 'there',
 'to',
 'took',
 'travelled',
 'up',
 'went',
 'yes'}

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer()
vocab_size = len(tokenizer.word_index) + 1
print("Vocab size:", vocab_size)

Vocab size: 1


In [ ]:
from keras.preprocessing.sequence import pad_sequences

In [ ]:
from keras.preprocessing import sequence
text = ['Mary moved to the bathroom . Sandra journeyed to the bedroom .']
tokenizer =Tokenizer(num_words=18)
tokenizer.fit_on_texts(text)
sequence = tokenizer.texts_to_sequences(text)
tokenizer = Tokenizer(filters=[])
tokenizer.fit_on_texts(vocab)
vocab_size = len(tokenizer.word_index) + 1 # Re-calculate vocab_size after tokenizer is fitted
tokenizer.word_index

{'the': 1,
 'hallway': 2,
 'travelled': 3,
 'bedroom': 4,
 'office': 5,
 'kitchen': 6,
 'picked': 7,
 'john': 8,
 'dropped': 9,
 'went': 10,
 'up': 11,
 'got': 12,
 'left': 13,
 'in': 14,
 'yes': 15,
 'sandra': 16,
 'to': 17,
 'journeyed': 18,
 'there': 19,
 'discarded': 20,
 'daniel': 21,
 'bathroom': 22,
 '.': 23,
 'back': 24,
 'put': 25,
 'milk': 26,
 'apple': 27,
 'moved': 28,
 '?': 29,
 'garden': 30,
 'grabbed': 31,
 'took': 32,
 'down': 33,
 'mary': 34,
 'is': 35,
 'no': 36,
 'football': 37}

In [ ]:
train_story_text = []
train_question_text = []
train_answers =[]
for story,question, answer in train_data:
  train_story_text.append(story)
  train_question_text.append(question)
train_story_seq = tokenizer.texts_to_sequences(train_story_text)
len(train_story_text)

10000

In [ ]:
len(train_story_seq)

10000

In [ ]:
def vectorize_stories(data, word_index=tokenizer.word_index, max_story_len=max_story_len, max_question_len=max_question_len):
  X = [] # This will collect all vectorized stories
  Xq = [] # This will collect all vectorized questions
  Y = [] # This will collect all one-hot encoded answers

  # In the tokenizer.word_index, index 0 is typically reserved for padding or unknown words.
  # We'll use 0 as the default value for words not found in the vocabulary.
  # This is consistent with how pad_sequences handles padding by default.

  for story, query, answer in data:
    # Process current story: map each word to its index, use 0 if word not found
    current_x = [word_index.get(word.lower(), 0) for word in story]
    X.append(current_x) # Append the vectorized current story to the list of all stories

    # Process current query: map each word to its index, use 0 if word not found
    current_xq = [word_index.get(word.lower(), 0) for word in query]
    Xq.append(current_xq) # Append the vectorized current query to the list of all questions

    # Process current answer: one-hot encode the answer
    # The size of the one-hot vector should be len(word_index) + 1 (for index 0).
    current_y_onehot = np.zeros(len(word_index) + 1)
    # Ensure the answer word is in word_index before accessing its index.
    # 'yes' and 'no' are guaranteed to be in vocab and thus in word_index.
    if answer.lower() in word_index:
        current_y_onehot[word_index[answer.lower()]] = 1
    Y.append(current_y_onehot) # Append the one-hot encoded current answer to the list of all answers

  # After processing all data, pad sequences and convert Y to a numpy array
  return (
      pad_sequences(X, maxlen=max_story_len),
      pad_sequences(Xq, maxlen=max_question_len),
      np.array(Y)
  )


In [ ]:
inputs_train, queries_train, answers_train = vectorize_stories(train_data)

In [ ]:
inputs_train.shape

(10000, 156)

In [ ]:
queries_train.shape

(10000, 6)

In [ ]:
answers_train.shape

(10000, 38)

In [ ]:
input_test, queries_test, answers_test = vectorize_stories(test_data)

In [ ]:
input_test

array([[ 0,  0,  0, ...,  1,  4, 23],
       [ 0,  0,  0, ...,  1, 30, 23],
       [ 0,  0,  0, ...,  1, 30, 23],
       ...,
       [ 0,  0,  0, ...,  1, 27, 23],
       [ 0,  0,  0, ...,  1, 30, 23],
       [ 0,  0,  0, ..., 27, 19, 23]], dtype=int32)

In [ ]:
queries_test

array([[35,  8, 14,  1,  6, 29],
       [35,  8, 14,  1,  6, 29],
       [35,  8, 14,  1, 30, 29],
       ...,
       [35, 34, 14,  1,  4, 29],
       [35, 16, 14,  1, 30, 29],
       [35, 34, 14,  1, 30, 29]], dtype=int32)

In [ ]:
answers_test

array([[0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [ ]:
tokenizer.word_index['no']

36

In [ ]:
from keras import Model
from keras.models import Sequential, Model
from keras.layers import Embedding
from keras.layers import Input, Activation, Dense, Permute, Dropout
from keras.layers import add, dot, concatenate
from keras.layers import LSTM

In [ ]:
input_sequence=Input((max_story_len,))
question= Input((max_question_len,))

In [ ]:
vocab_size = len(tokenizer.word_index) + 1

max_len_input = inputs_train.shape[1]
max_len_query = queries_train.shape[1]

# STORY INPUT
input_story = Input(shape=(max_len_input,))
story_embed = Embedding(vocab_size, 64)(input_story)
story_vec = LSTM(32)(story_embed)

# QUESTION INPUT
input_question = Input(shape=(max_len_query,))
question_embed = Embedding(vocab_size, 64)(input_question)
question_vec = LSTM(32)(question_embed)

# MERGE
merged = Concatenate()([story_vec, question_vec])

# OUTPUT
output = Dense(vocab_size, activation='softmax')(merged)

model = Model([input_story, input_question], output)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_17"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_19      │ (None, 156)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_20      │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_17        │ (None, 156, 64)   │      2,432 │ input_layer_19[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_18        │ (None, 6, 64)     │      2,432 │ input_layer_20[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_15 (LSTM)      │ (None, 32)        │     12,416 │ embedding_17[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_16 (LSTM)      │ (None, 32)        │     12,416 │ embedding_18[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_12      │ (None, 64)        │          0 │ lstm_15[0][0],    │
│ (Concatenate)       │                   │            │ lstm_16[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 38)        │      2,470 │ concatenate_12[0… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,166 (125.65 KB)

 Trainable params: 32,166 (125.65 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    [inputs_train, queries_train],
    answers_train,
    batch_size=32,
    epochs=30,
    validation_data=([input_test, queries_test], answers_test)
)

Epoch 1/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 32s 80ms/step - accuracy: 0.4833 - loss: 1.3923 - val_accuracy: 0.5030 - val_loss: 0.6964
Epoch 2/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 49s 106ms/step - accuracy: 0.4933 - loss: 0.6987 - val_accuracy: 0.5030 - val_loss: 0.7046
Epoch 3/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 84ms/step - accuracy: 0.4832 - loss: 0.7001 - val_accuracy: 0.4970 - val_loss: 0.7088
Epoch 4/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 84ms/step - accuracy: 0.5042 - loss: 0.6962 - val_accuracy: 0.5110 - val_loss: 0.6932
Epoch 5/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 25s 80ms/step - accuracy: 0.5107 - loss: 0.6965 - val_accuracy: 0.5030 - val_loss: 0.6934
Epoch 6/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 28s 89ms/step - accuracy: 0.5055 - loss: 0.6945 - val_accuracy: 0.4970 - val_loss: 0.6994
Epoch 7/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 84ms/step - accuracy: 0.4899 - loss: 0.6959 - val_accuracy: 0.5110 - val_loss: 0.6926
Epoch 8/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 24s 77ms/step - accuracy: 0.4942 - loss: 0.6963 -

In [138]:
pred_results=model.predict(([input_test, queries_test]))

32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 90ms/step


In [139]:
story = ' '.join(word for word in test_data[0][0])
print(story)

Mary got the milk there . John moved to the bedroom .


In [140]:
query = ' '.join(word for word in test_data[0][1])
print(query)

Is John in the kitchen ?


In [141]:
print("True Test Answer from Data is:",test_data[0][2])

True Test Answer from Data is: no


In [142]:
val_max=np.argmax(pred_results[0])

for key, val in tokenizer.word_index.items():
  if val==val_max:
    k=key
print("Predited answer is: ",k)
print("Probability of certainity was: ", pred_results[0][val_max])

Predited answer is:  no
Probability of certainity was:  0.50138974


In [143]:
my_story='John travelled the bathroom last saturday. Sandra dropped the football in the garden.'
my_story

'John travelled the bathroom last saturday. Sandra dropped the football in the garden.'

In [144]:
my_question="Did John travelled the bathroom last friday?"

In [145]:
my_question.split()

['Did', 'John', 'travelled', 'the', 'bathroom', 'last', 'friday?']

In [146]:
mydata=[(my_story,my_question,'yes')]

In [147]:
my_story, my_ques, my_ans= vectorize_stories(mydata)

In [148]:
pred_results=model.predict(([my_story,my_ques]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
